# Answer

Using the vector db created from `ingest.py` we answer queries

In [ ]:
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [4]:
VECTORDB_PATH = "../irc_vectordb"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
GPT_MODEL = "gpt-4.1-nano"
load_dotenv(override=True)

True

In [6]:
embeddings = HuggingFaceEmbeddings(model=EMBEDDING_MODEL)
vectordb = Chroma(persist_directory=VECTORDB_PATH, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
retriever = vectordb.as_retriever()

In [ ]:
retriever.invoke("What is section 1031?")

[Document(id='a00eab19-6b4e-45fa-bf6d-9dccba898015', metadata={'trapped': '', 'file_path': '../Internal Revenue Code/Internal Revenue Code.pdf', 'modDate': "D:20260112072823-05'00'", 'page': 872, 'title': '', 'total_pages': 7164, 'creationDate': "D:20260112072823-05'00'", 'format': 'PDF 1.4', 'subject': '', 'producer': 'iText 2.0.8 (by lowagie.com)', 'moddate': '2026-01-12T07:28:23-05:00', 'source': '../Internal Revenue Code/Internal Revenue Code.pdf', 'author': '', 'creationdate': '2026-01-12T07:28:23-05:00', 'creator': '', 'keywords': ''}, page_content='1034 was repealed by Pub. L. 105–34, title III, §312(b), Aug. 5, 1997, 111 Stat. 839.\nThe Robert T. Stafford Disaster Relief and Emergency Assistance Act, referred to in subsec. (t)(11)(E),\n(F)(i)(I), is Pub. L. 93–288, May 22, 1974, 88 Stat. 143, which is classified principally to chapter 68 (§5121 et\nseq.) of Title 42, The Public Health and Welfare. Section 401 of the Act is classified to section 5170 of Title\n42. For complete c

In [ ]:
retriever.invoke("§ 1031") # § = "section" in the IRC pdf

[Document(id='ee6a4019-7fd0-4737-b5ca-4e6ee1a2ff12', metadata={'page': 1674, 'total_pages': 7164, 'format': 'PDF 1.4', 'producer': 'iText 2.0.8 (by lowagie.com)', 'trapped': '', 'creator': '', 'keywords': '', 'subject': '', 'source': '../Internal Revenue Code/Internal Revenue Code.pdf', 'creationdate': '2026-01-12T07:28:23-05:00', 'modDate': "D:20260112072823-05'00'", 'moddate': '2026-01-12T07:28:23-05:00', 'file_path': '../Internal Revenue Code/Internal Revenue Code.pdf', 'creationDate': "D:20260112072823-05'00'", 'title': '', 'author': ''}, page_content='Subsec. (b)(6). Pub. L. 94–455, §1501(b)(4)(B), added par. (6).\nSubsec. (c)(2). Pub. L. 94–455, §1501(b)(4)(C), inserted "For purposes of this section, the determination of\nwhether an individual is married shall be made in accordance with the provisions of section 143(a)" after\n"community property laws".\nSubsec. (c)(3). Pub. L. 94–455, §1501(b)(4)(D), added par. (3).\nSubsec. (c)(4). Pub. L. 94–455, §1503(a), added par. (4).\nSTA

Proper documents arent being retrieved

We need to try a different strategy so the proper documents are retrieved

I changed the Ingest strategy.
1. I am using a different file format
2. I cleaned the data before saving to vectordb

Lets see how it is now

In [ ]:
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

VECTORDB_PATH = "../irc_xml_vectordb"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
GPT_MODEL = "gpt-4.1-nano"
load_dotenv(override=True)

True

In [12]:
embeddings = HuggingFaceEmbeddings(model=EMBEDDING_MODEL)
vectordb = Chroma(persist_directory=VECTORDB_PATH, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
retriever = vectordb.as_retriever()

In [14]:
retriever.invoke("What is section 1031")

[Document(id='883385c9-bf0b-4803-b86b-71adc6c87359', metadata={'subpart': '', 'identifier': '/us/usc/t26/s3401', 'chapter': 'CHAPTER 24— COLLECTION OF INCOME TAX AT SOURCE ON WAGES', 'part': '', 'subtitle': 'Subtitle C— Employment Taxes', 'heading': 'Definitions', 'section': '3401', 'subchapter': '', 'source': 'IRC'}, page_content='§ 3401'),
 Document(id='56032822-4316-4b45-98bd-a77d250bf1fb', metadata={'heading': 'Designation procedure', 'part': 'PART I— DESIGNATION', 'identifier': '/us/usc/t26/s1391', 'source': 'IRC', 'subtitle': 'Subtitle A— Income Taxes', 'section': '1391', 'chapter': 'CHAPTER 1— NORMAL TAXES AND SURTAXES', 'subchapter': 'Subchapter U— Designation and Treatment of Empowerment Zones, Enterprise Communities, and Rural Development Investment Areas', 'subpart': ''}, page_content='§ 1391 Designation procedure'),
 Document(id='4f45c871-97b9-433c-8f9f-763258e9d888', metadata={'subchapter': 'Subchapter G— Breweries', 'chapter': 'CHAPTER 51— DISTILLED SPIRITS, WINES, AND BE

Well that didnt work as well as I wouldv'e hoped. Lets try something else

In [32]:
import re
import json
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.schema import Document

VECTORDB_PATH = "../irc_xml_vectordb"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
GPT_MODEL = "gpt-4.1-nano"
load_dotenv(override=True)

True

In [33]:
def load_chunks(input_path="../Internal Revenue Code/irc_chunks.json"):
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    chunks = [Document(page_content=d["text"], metadata=d["metadata"]) for d in data]
    print(f"Loaded {len(chunks)} chunks from {input_path}")
    return chunks

chunks = load_chunks()

# Vector Retriever
embeddings = HuggingFaceEmbeddings(model=EMBEDDING_MODEL)
vectordb = Chroma(persist_directory=VECTORDB_PATH, embedding_function=embeddings)

vector_retriever = vectordb.as_retriever()

# BM25 Retriever 
bm25_retriever = BM25Retriever.from_documents(chunks)

# Ensemble Retriever 

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.6, 0.4]   # weight BM25 higher for section number queries
)

Loaded 4011 chunks from ../Internal Revenue Code/irc_chunks.json


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [34]:
def test_retrieval(query):
    print(f"\nQuery: '{query}'")
    print("=" * 60)

    # BM25 alone
    bm25_results = bm25_retriever.invoke(query)
    print("\nBM25 results:")
    for doc in bm25_results:
        print(f"  § {doc.metadata['section']} — {doc.metadata['heading']}")

    # Vector alone  
    vector_results = vector_retriever.invoke(query)
    print("\nVector results:")
    for doc in vector_results:
        print(f"  § {doc.metadata['section']} — {doc.metadata['heading']}")

    # Combined
    ensemble_results = ensemble_retriever.invoke(query)
    print("\nEnsemble results:")
    for doc in ensemble_results:
        print(f"  § {doc.metadata['section']} — {doc.metadata['heading']}")

test_retrieval("what does section 1 state?")


Query: 'what does section 1 state?'

BM25 results:
  § 6103 — Confidentiality and disclosure of returns and return information
  § 139E — Indian general welfare benefits
  § 1388 — Definitions; special rules
  § 6702 — Frivolous tax submissions

Vector results:
  § 1392 — Eligibility criteria
  § 4980C — Requirements for issuers of qualified long-term care insurance contracts
  § 45D — New markets tax credit
  § 911 — Citizens or residents of the United States living abroad

Ensemble results:
  § 6103 — Confidentiality and disclosure of returns and return information
  § 139E — Indian general welfare benefits
  § 1388 — Definitions; special rules
  § 6702 — Frivolous tax submissions
  § 1392 — Eligibility criteria
  § 4980C — Requirements for issuers of qualified long-term care insurance contracts
  § 45D — New markets tax credit
  § 911 — Citizens or residents of the United States living abroad


It's not retrieving relevant chunks - Trying a new strategy

In [41]:
def smart_retrieve(query, vectorstore, ensemble_retriever, k=6):
    """
    If the query mentions a specific section number, bypass retrieval
    entirely and pull directly from vectorstore using metadata filter.
    Otherwise fall back to ensemble retrieval.
    """
    # Detect section number in query
    # Matches: "section 1031", "§ 1031", "sec 1031", "1031" alone
    match = re.search(
        r'(?:section|sec|§)\s*(\d+[A-Z]?)|^(\d+[A-Z]?)$',
        query.strip(),
        re.IGNORECASE
    )

    if match:
        section_num = (match.group(1) or match.group(2)).upper()
        print(f"  Section detected: § {section_num} — using metadata filter")

        results = vectorstore.get(where={"section": section_num})

        if results and results["documents"]:
            print(f"  Found {len(results['documents'])} chunks for § {section_num}")
            return [
                Document(page_content=doc, metadata=meta)
                for doc, meta in zip(results["documents"], results["metadatas"])
            ]
        else:
            print(f"  § {section_num} not found in vectorstore — falling back to ensemble")

    # No section number — use ensemble
    print("  No section detected — using ensemble retrieval")
    return ensemble_retriever.invoke(query)


# Test it
queries = [
    # "what is section 1031",
    # "§ 1031",
    # "like kind exchange",
    #"what is the capital gains tax rate",
    "section 401k retirement plans",
]

for q in queries:
    print(f"\nQuery: '{q}'")
    print("=" * 50)
    results = smart_retrieve(q, vectordb, ensemble_retriever)
    for doc in results:
        # print(f"  § {doc.metadata['section']} — {doc.metadata['heading']}")
        print(f"  {doc.page_content}")
        print()


Query: 'section 401k retirement plans'
  Section detected: § 401K — using metadata filter
  § 401K not found in vectorstore — falling back to ensemble
  No section detected — using ensemble retrieval
  § 401 Qualified pension, profit-sharing, and stock bonus plans
. For purposes of this clause, the employer-derived retirement benefit created under Federal law shall be treated as accruing ratably over 35 years. (ii) Final pay .— For purposes of this subparagraph, the participant’s final pay is the compensation (as defined in section 414(q)(4)) paid to the participant by the employer for any year— (I) which ends during the 5-year period ending with the year in which the participant separated from service for the employer, and (II) for which the participant’s total compensation from the employer was highest. (E) 2 or more plans treated as single plan .— For purposes of determining whether 2 or more plans of an employer satisfy the requirements of paragraph (4) when considered as a single